## Final test set values on best models

In [7]:
import pandas as pd
import itertools
import numpy as np
import argparse
from pyhere import here
import os
import pickle
from plotnine import *
from sklearn.model_selection import train_test_split
from sklearn import set_config
from skopt import BayesSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from skopt.space import Categorical, Real, Integer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score, cross_validate, KFold
from sklearn.metrics import root_mean_squared_error, r2_score

set_config(transform_output="pandas")

In [2]:
class RatioGenerator(BaseEstimator, TransformerMixin):
    '''
    A custom transformer that generates new features by taking the ratios of all combinations of specified columns.
    For use with the flux columns
    '''
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        # add a dummy attribute so sklearn knows this transformer is fitted
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        # Create a copy to avoid SettingWithCopy warnings or mutating the original
        X_out = X.copy()
        
        for top, bottom in itertools.combinations(self.cols, 2):
            new_col_name = f"{top}_over_{bottom}"
            X_out[new_col_name] = X_out[top] / X_out[bottom]
            
        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            raise ValueError("input_features must be provided")

        input_features = list(input_features)

        # Validate columns exist
        missing = set(self.cols) - set(input_features)
        if missing:
            raise ValueError(f"Missing columns in input_features: {missing}")

        # Generate ratio feature names
        ratio_features = [
            f"{top}_over_{bottom}"
            for top, bottom in itertools.combinations(self.cols, 2)
        ]

        # IMPORTANT: include ALL original input features
        return np.array(input_features + ratio_features, dtype=object)

In [3]:
phot = pd.read_csv(here("data/cleaned", "MIRION_cleaned_everything.csv"))

flux_cols = ['F1100', 'F870', 'F500', 'F350', 'F250', 'F160', 'F70', 'F24', 'F12', 'F8']

cv = KFold(n_splits=10)

## Temperature

In [4]:
with open(here("pipeline/results", "catlogratio_TEMP_results.pkl"), 'rb') as file:
    temp_results = pickle.load(file)

temp_results['best_params']

OrderedDict([('impute', SimpleImputer(strategy='median')),
             ('model__bagging_temperature', 10.0),
             ('model__border_count', 255),
             ('model__colsample_bylevel', 1.0),
             ('model__depth', 3),
             ('model__grow_policy', 'Lossguide'),
             ('model__iterations', 3000),
             ('model__l2_leaf_reg', 21.671078923255866),
             ('model__learning_rate', 0.05518369736341342),
             ('model__min_data_in_leaf', 100),
             ('model__random_strength', 1e-09),
             ('scale', StandardScaler())])

In [5]:
y_temp = phot['TEMP']
X_temp = phot.drop(columns=['LRATIO', 'T_BOL', 'LM', 'L_BOL', 'MASS', 'DIAM', 'SURF_DENS', 'YB'])

temp_X_train, temp_X_test, temp_y_train, temp_y_test = train_test_split(X_temp, y_temp, test_size = 0.2, random_state=2026)

best_temp_model = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('log', ColumnTransformer([('log_ratio', FunctionTransformer(func=np.log), ['F1100_over_F870', 'F1100_over_F500', 'F1100_over_F350', 'F1100_over_F250', 'F1100_over_F160', 'F1100_over_F70', 'F1100_over_F24', 'F1100_over_F12', 'F1100_over_F8', 'F870_over_F500', 'F870_over_F350', 'F870_over_F250', 'F870_over_F160', 'F870_over_F70', 'F870_over_F24', 'F870_over_F12', 'F870_over_F8', 'F500_over_F350', 'F500_over_F250', 'F500_over_F160', 'F500_over_F70', 'F500_over_F24', 'F500_over_F12', 'F500_over_F8', 'F350_over_F250', 'F350_over_F160', 'F350_over_F70', 'F350_over_F24', 'F350_over_F12', 'F350_over_F8', 'F250_over_F160', 'F250_over_F70', 'F250_over_F24', 'F250_over_F12', 'F250_over_F8', 'F160_over_F70', 'F160_over_F24', 'F160_over_F12', 'F160_over_F8', 'F70_over_F24', 'F70_over_F12', 'F70_over_F8', 'F24_over_F12', 'F24_over_F8', 'F12_over_F8'])], remainder='passthrough')),
    ('scale', StandardScaler()),
    ('model', CatBoostRegressor(random_state=2026, verbose=0, thread_count=-1, loss_function='RMSE'))
])

best_temp_model.set_params(
model__bagging_temperature=10.0,
model__border_count=255,
model__colsample_bylevel=1.0,
model__depth=3,
model__grow_policy='Lossguide',
model__iterations=3000,
model__l2_leaf_reg=21.671078923255866,
model__learning_rate=0.05518369736341342,
model__min_data_in_leaf=100,
model__random_strength=1e-09,
)


best_temp_model_cv = cross_val_score(best_temp_model, temp_X_train, temp_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(temp_results['CVscore'], -best_temp_model_cv.mean())

0.01865037068828776 0.01865037068828776


In [6]:
best_temp_model.fit(temp_X_train, temp_y_train)
temp_y_preds = best_temp_model.predict(temp_X_test)
temp_rmse = root_mean_squared_error(temp_y_test, temp_y_preds)
temp_r2 = r2_score(temp_y_test, temp_y_preds)
print(temp_rmse, temp_r2)

0.026825991724962445 0.9999687577232964


In [17]:
temp_importances = best_temp_model.named_steps['model'].feature_importances_
temp_imp_df = pd.DataFrame({
    'Feature': best_temp_model.named_steps['model'].feature_names_,
    'Importance': temp_importances
}).sort_values(by='Importance', ascending=False)
temp_imp_df.head()

# (
#     ggplot(temp_imp_df, aes(x='Feature', y='Importance')) +
#     geom_bar()
# )

,Feature,Importance
45,remainder__TEMP,99.999726
46,remainder__F160,0.000046
38,log_ratio__F160_over_F8,0.000041
25,log_ratio__F350_over_F160,0.000038
35,log_ratio__F160_over_F70,0.000025


## Surface Density (log scale because distribution is heavily skewed)

In [23]:
with open(here("pipeline/results", "catlogratio_SURF_DENS_results.pkl"), 'rb') as file:
    dens_results = pickle.load(file)

dens_results['logbest_params']

OrderedDict([('impute', SimpleImputer()),
             ('model__bagging_temperature', 10.0),
             ('model__border_count', 255),
             ('model__colsample_bylevel', 1.0),
             ('model__depth', 12),
             ('model__grow_policy', 'Lossguide'),
             ('model__iterations', 1787),
             ('model__l2_leaf_reg', 0.001),
             ('model__learning_rate', 0.05754542044564178),
             ('model__min_data_in_leaf', 100),
             ('model__random_strength', 2.529193345341249e-07),
             ('scale', RobustScaler())])

In [ ]:
y_log_dens = np.log(phot['SURF_DENS'])
X_dens = phot.drop(columns=['LRATIO', 'T_BOL', 'LM', 'L_BOL', 'MASS', 'DIAM', 'TEMP', 'YB'])

dens_X_train, dens_X_test, dens_y_train, dens_y_test = train_test_split(X_dens, y_log_dens, test_size = 0.2, random_state=2026)

best_dens_model = Pipeline([
    ('impute', SimpleImputer()),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('log', ColumnTransformer([('log_ratio', FunctionTransformer(func=np.log), ['F1100_over_F870', 'F1100_over_F500', 'F1100_over_F350', 'F1100_over_F250', 'F1100_over_F160', 'F1100_over_F70', 'F1100_over_F24', 'F1100_over_F12', 'F1100_over_F8', 'F870_over_F500', 'F870_over_F350', 'F870_over_F250', 'F870_over_F160', 'F870_over_F70', 'F870_over_F24', 'F870_over_F12', 'F870_over_F8', 'F500_over_F350', 'F500_over_F250', 'F500_over_F160', 'F500_over_F70', 'F500_over_F24', 'F500_over_F12', 'F500_over_F8', 'F350_over_F250', 'F350_over_F160', 'F350_over_F70', 'F350_over_F24', 'F350_over_F12', 'F350_over_F8', 'F250_over_F160', 'F250_over_F70', 'F250_over_F24', 'F250_over_F12', 'F250_over_F8', 'F160_over_F70', 'F160_over_F24', 'F160_over_F12', 'F160_over_F8', 'F70_over_F24', 'F70_over_F12', 'F70_over_F8', 'F24_over_F12', 'F24_over_F8', 'F12_over_F8'])], remainder='passthrough')),
    ('scale', RobustScaler()),
    ('model', CatBoostRegressor(random_state=2026, verbose=0, thread_count=-1, loss_function='RMSE'))
])

best_dens_model.set_params(
model__bagging_temperature=10,
model__border_count=225,
model__colsample_bylevel=1.0,
model__depth=12,
model__grow_policy='Lossguide',
model__iterations=1787,
model__l2_leaf_reg=0.001,
model__learning_rate=0.05754542044564178,
model__min_data_in_leaf=100,
model__random_strength=2.529193345341249e-07
)

best_dens_model_cv = cross_val_score(best_dens_model, dens_X_train, dens_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(dens_results['logCVscore'], -best_dens_model_cv.mean())

0.0048819896417874025 0.005416015632899092


In [26]:
best_dens_model.fit(dens_X_train, dens_y_train)
dens_y_preds = best_dens_model.predict(dens_X_test)
dens_rmse = root_mean_squared_error(dens_y_test, dens_y_preds)
dens_r2 = r2_score(dens_y_test, dens_y_preds)
print(dens_rmse, dens_r2)

0.010417586121982854 0.9999397109853716


## Bolometric Temperature

In [27]:
with open(here("pipeline/results", "rflogratio_T_BOL_results.pkl"), 'rb') as file:
    tbol_results = pickle.load(file)

tbol_results['best_params']

OrderedDict([('impute', SimpleImputer(strategy='median')),
             ('model__bootstrap', True),
             ('model__max_depth', 11),
             ('model__max_features', None),
             ('model__min_samples_leaf', 3),
             ('model__min_samples_split', 2),
             ('model__n_estimators', 778),
             ('scale', 'passthrough')])

In [34]:
y_tbol = phot['T_BOL']
X_tbol = phot.drop(columns=['LRATIO', 'SURF_DENS', 'LM', 'L_BOL', 'MASS', 'DIAM', 'TEMP', 'YB'])

tbol_X_train, tbol_X_test, tbol_y_train, tbol_y_test = train_test_split(X_tbol, y_tbol, test_size = 0.2, random_state=2026)

best_tbol_model = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('log', ColumnTransformer([('log_ratio', FunctionTransformer(func=np.log), ['F1100_over_F870', 'F1100_over_F500', 'F1100_over_F350', 'F1100_over_F250', 'F1100_over_F160', 'F1100_over_F70', 'F1100_over_F24', 'F1100_over_F12', 'F1100_over_F8', 'F870_over_F500', 'F870_over_F350', 'F870_over_F250', 'F870_over_F160', 'F870_over_F70', 'F870_over_F24', 'F870_over_F12', 'F870_over_F8', 'F500_over_F350', 'F500_over_F250', 'F500_over_F160', 'F500_over_F70', 'F500_over_F24', 'F500_over_F12', 'F500_over_F8', 'F350_over_F250', 'F350_over_F160', 'F350_over_F70', 'F350_over_F24', 'F350_over_F12', 'F350_over_F8', 'F250_over_F160', 'F250_over_F70', 'F250_over_F24', 'F250_over_F12', 'F250_over_F8', 'F160_over_F70', 'F160_over_F24', 'F160_over_F12', 'F160_over_F8', 'F70_over_F24', 'F70_over_F12', 'F70_over_F8', 'F24_over_F12', 'F24_over_F8', 'F12_over_F8'])], remainder='passthrough')),
    ('scale', 'passthrough'),
    ('model', RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1))
])

best_tbol_model.set_params(
model__bootstrap=True,
model__max_depth=11,
model__max_features=None,
model__min_samples_leaf=3,
model__min_samples_split=2,
model__n_estimators=778
)

best_tbol_model_cv = cross_val_score(best_tbol_model, tbol_X_train, tbol_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(tbol_results['CVscore'], -best_tbol_model_cv.mean())

0.4328639073234578 0.43286390732345714


In [35]:
best_tbol_model.fit(tbol_X_train, tbol_y_train)
tbol_y_preds = best_tbol_model.predict(tbol_X_test)
tbol_rmse = root_mean_squared_error(tbol_y_test, tbol_y_preds)
tbol_r2 = r2_score(tbol_y_test, tbol_y_preds)
print(tbol_rmse, tbol_r2)

0.24947346517683183 0.9995542320855707


## Luminosity Mass Ratio (log scale because significantly right skewed)

In [36]:
with open(here("pipeline/results", "rflogratio_LM_results.pkl"), 'rb') as file:
    lm_results = pickle.load(file)

lm_results['logbest_params']

OrderedDict([('impute', SimpleImputer(strategy='median')),
             ('model__bootstrap', True),
             ('model__max_depth', 14),
             ('model__max_features', None),
             ('model__min_samples_leaf', 1),
             ('model__min_samples_split', 2),
             ('model__n_estimators', 38),
             ('scale', StandardScaler())])

In [ ]:
y_log_lm = np.log(phot['LM'])
X_lm = phot.drop(columns=['LRATIO', 'SURF_DENS', 'T_BOL', 'L_BOL', 'MASS', 'DIAM', 'TEMP', 'YB'])

lm_X_train, lm_X_test, lm_y_train, lm_y_test = train_test_split(X_lm, y_log_lm, test_size = 0.2, random_state=2026)

best_lm_model = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('log', ColumnTransformer([('log_ratio', FunctionTransformer(func=np.log), ['F1100_over_F870', 'F1100_over_F500', 'F1100_over_F350', 'F1100_over_F250', 'F1100_over_F160', 'F1100_over_F70', 'F1100_over_F24', 'F1100_over_F12', 'F1100_over_F8', 'F870_over_F500', 'F870_over_F350', 'F870_over_F250', 'F870_over_F160', 'F870_over_F70', 'F870_over_F24', 'F870_over_F12', 'F870_over_F8', 'F500_over_F350', 'F500_over_F250', 'F500_over_F160', 'F500_over_F70', 'F500_over_F24', 'F500_over_F12', 'F500_over_F8', 'F350_over_F250', 'F350_over_F160', 'F350_over_F70', 'F350_over_F24', 'F350_over_F12', 'F350_over_F8', 'F250_over_F160', 'F250_over_F70', 'F250_over_F24', 'F250_over_F12', 'F250_over_F8', 'F160_over_F70', 'F160_over_F24', 'F160_over_F12', 'F160_over_F8', 'F70_over_F24', 'F70_over_F12', 'F70_over_F8', 'F24_over_F12', 'F24_over_F8', 'F12_over_F8'])], remainder='passthrough')),
    ('scale', StandardScaler()),
    ('model', RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1))
])

best_lm_model.set_params(
model__bootstrap=True,
model__max_depth=14,
model__max_features=None,
model__min_samples_leaf=1,
model__min_samples_split=2,
model__n_estimators=38
)

best_lm_model_cv = cross_val_score(best_lm_model, lm_X_train, lm_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(lm_results['logCVscore'], -best_lm_model_cv.mean())

0.01708851509754606 0.01719479601293345


In [38]:
best_lm_model.fit(lm_X_train, lm_y_train)
lm_y_preds = best_lm_model.predict(lm_X_test)
lm_rmse = root_mean_squared_error(lm_y_test, lm_y_preds)
lm_r2 = r2_score(lm_y_test, lm_y_preds)
print(lm_rmse, lm_r2)

0.007648980746330525 0.9999703161703211


## Luminosity ratio (logged because distribution)

In [5]:
with open(here("pipeline/results", "rflogratio_LRATIO_results.pkl"), 'rb') as file:
    lrat_results = pickle.load(file)

lrat_results['logbest_params']

OrderedDict([('impute', SimpleImputer()),
             ('model__bootstrap', True),
             ('model__max_depth', 25),
             ('model__max_features', None),
             ('model__min_samples_leaf', 1),
             ('model__min_samples_split', 2),
             ('model__n_estimators', 1000),
             ('scale', RobustScaler())])

In [7]:
y_log_lrat = np.log(phot['LRATIO'])
X_lrat = phot.drop(columns=['LM', 'SURF_DENS', 'T_BOL', 'L_BOL', 'MASS', 'DIAM', 'TEMP', 'YB'])

lrat_X_train, lrat_X_test, lrat_y_train, lrat_y_test = train_test_split(X_lrat, y_log_lrat, test_size = 0.2, random_state=2026)

best_lrat_model = Pipeline([
    ('impute', SimpleImputer()),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('log', ColumnTransformer([('log_ratio', FunctionTransformer(func=np.log), ['F1100_over_F870', 'F1100_over_F500', 'F1100_over_F350', 'F1100_over_F250', 'F1100_over_F160', 'F1100_over_F70', 'F1100_over_F24', 'F1100_over_F12', 'F1100_over_F8', 'F870_over_F500', 'F870_over_F350', 'F870_over_F250', 'F870_over_F160', 'F870_over_F70', 'F870_over_F24', 'F870_over_F12', 'F870_over_F8', 'F500_over_F350', 'F500_over_F250', 'F500_over_F160', 'F500_over_F70', 'F500_over_F24', 'F500_over_F12', 'F500_over_F8', 'F350_over_F250', 'F350_over_F160', 'F350_over_F70', 'F350_over_F24', 'F350_over_F12', 'F350_over_F8', 'F250_over_F160', 'F250_over_F70', 'F250_over_F24', 'F250_over_F12', 'F250_over_F8', 'F160_over_F70', 'F160_over_F24', 'F160_over_F12', 'F160_over_F8', 'F70_over_F24', 'F70_over_F12', 'F70_over_F8', 'F24_over_F12', 'F24_over_F8', 'F12_over_F8'])], remainder='passthrough')),
    ('scale', RobustScaler()),
    ('model', RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1))
])

best_lrat_model.set_params(
model__bootstrap=True,
model__max_depth=25,
model__max_features=None,
model__min_samples_leaf=1,
model__min_samples_split=2,
model__n_estimators=1000
)

best_lrat_model_cv = cross_val_score(best_lrat_model, lrat_X_train, lrat_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(lrat_results['logCVscore'], -best_lrat_model_cv.mean())

0.02827558612273919 0.028384809128559462


In [8]:
best_lrat_model.fit(lrat_X_train, lrat_y_train)
lrat_y_preds = best_lrat_model.predict(lrat_X_test)
lrat_rmse = root_mean_squared_error(lrat_y_test, lrat_y_preds)
lrat_r2 = r2_score(lrat_y_test, lrat_y_preds)
print(lrat_rmse, lrat_r2)

0.03165270850293977 0.9993285647985254


## Diameter (logged because more useful-ish)

In [21]:
with open(here("pipeline/results", "rf_DIAM_results.pkl"), 'rb') as file:
    diam_results = pickle.load(file)

diam_results['logbest_params']

OrderedDict([('impute', SimpleImputer()),
             ('model__bootstrap', False),
             ('model__max_depth', 20),
             ('model__max_features', None),
             ('model__min_samples_leaf', 3),
             ('model__min_samples_split', 2),
             ('model__n_estimators', 273),
             ('scale', RobustScaler())])

In [ ]:
y_log_diam = np.log(phot['DIAM'])
X_diam = phot.drop(columns=['LM', 'SURF_DENS', 'T_BOL', 'L_BOL', 'MASS', 'LRATIO', 'TEMP', 'YB'])

diam_X_train, diam_X_test, diam_y_train, diam_y_test = train_test_split(X_diam, y_log_diam, test_size = 0.2, random_state=2026)

best_diam_model = Pipeline([
    ('impute', SimpleImputer()),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('scale', RobustScaler()),
    ('model', RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1))
    ])

best_diam_model.set_params(
model__bootstrap=False,
model__max_depth=20,
model__max_features=None,
model__min_samples_leaf=3,
model__min_samples_split=2,
model__n_estimators=273
)

best_diam_model_cv = cross_val_score(best_diam_model, diam_X_train, diam_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(diam_results['logCVscore'], -best_diam_model_cv.mean())

0.01314176606691626 0.013250404838057945


In [23]:
best_diam_model.fit(diam_X_train, diam_y_train)
diam_y_preds = best_diam_model.predict(diam_X_test)
diam_rmse = root_mean_squared_error(diam_y_test, diam_y_preds)
diam_r2 = r2_score(diam_y_test, diam_y_preds)
print(diam_rmse, diam_r2)

0.013804568087981781 0.9996897389476936


## Mass (log scale bc stellar mass function)

In [14]:
with open(here("pipeline/results", "rflogratio_MASS_results.pkl"), 'rb') as file:
    mass_results = pickle.load(file)

mass_results['logbest_params']

OrderedDict([('impute', KNNImputer()),
             ('model__bootstrap', True),
             ('model__max_depth', 13),
             ('model__max_features', None),
             ('model__min_samples_leaf', 2),
             ('model__min_samples_split', 2),
             ('model__n_estimators', 648),
             ('scale', RobustScaler())])

In [15]:
y_log_mass = np.log(phot['MASS'])
X_mass = phot.drop(columns=['LM', 'SURF_DENS', 'T_BOL', 'L_BOL', 'LRATIO', 'DIAM', 'TEMP', 'YB'])

mass_X_train, mass_X_test, mass_y_train, mass_y_test = train_test_split(X_mass, y_log_mass, test_size = 0.2, random_state=2026)

best_mass_model = Pipeline([
    ('impute', KNNImputer()),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('log', ColumnTransformer([('log_ratio', FunctionTransformer(func=np.log), ['F1100_over_F870', 'F1100_over_F500', 'F1100_over_F350', 'F1100_over_F250', 'F1100_over_F160', 'F1100_over_F70', 'F1100_over_F24', 'F1100_over_F12', 'F1100_over_F8', 'F870_over_F500', 'F870_over_F350', 'F870_over_F250', 'F870_over_F160', 'F870_over_F70', 'F870_over_F24', 'F870_over_F12', 'F870_over_F8', 'F500_over_F350', 'F500_over_F250', 'F500_over_F160', 'F500_over_F70', 'F500_over_F24', 'F500_over_F12', 'F500_over_F8', 'F350_over_F250', 'F350_over_F160', 'F350_over_F70', 'F350_over_F24', 'F350_over_F12', 'F350_over_F8', 'F250_over_F160', 'F250_over_F70', 'F250_over_F24', 'F250_over_F12', 'F250_over_F8', 'F160_over_F70', 'F160_over_F24', 'F160_over_F12', 'F160_over_F8', 'F70_over_F24', 'F70_over_F12', 'F70_over_F8', 'F24_over_F12', 'F24_over_F8', 'F12_over_F8'])], remainder='passthrough')),
    ('scale', RobustScaler()),
    ('model', RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1))
])

best_mass_model.set_params(
model__bootstrap=True,
model__max_depth=13,
model__max_features=None,
model__min_samples_leaf=2,
model__min_samples_split=2,
model__n_estimators=648
)

best_mass_model_cv = cross_val_score(best_mass_model, mass_X_train, mass_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(mass_results['logCVscore'], -best_mass_model_cv.mean())

0.03813400759600334 0.038136605787176286


In [16]:
best_mass_model.fit(mass_X_train, mass_y_train)
mass_y_preds = best_mass_model.predict(mass_X_test)
mass_rmse = root_mean_squared_error(mass_y_test, mass_y_preds)
mass_r2 = r2_score(mass_y_test, mass_y_preds)
print(mass_rmse, mass_r2)

0.03426477206256758 0.9996924378182173


## Bolometric Luminosity (log scale bc dist)

In [17]:
with open(here("pipeline/results", "rflogratio_L_BOL_results.pkl"), 'rb') as file:
    lbol_results = pickle.load(file)

lbol_results['logbest_params']

OrderedDict([('impute', KNNImputer()),
             ('model__bootstrap', True),
             ('model__max_depth', 30),
             ('model__max_features', None),
             ('model__min_samples_leaf', 2),
             ('model__min_samples_split', 2),
             ('model__n_estimators', 1000),
             ('scale', RobustScaler())])

In [18]:
y_log_lbol = np.log(phot['L_BOL'])
X_lbol = phot.drop(columns=['LM', 'SURF_DENS', 'T_BOL', 'MASS', 'LRATIO', 'DIAM', 'TEMP', 'YB'])

lbol_X_train, lbol_X_test, lbol_y_train, lbol_y_test = train_test_split(X_lbol, y_log_lbol, test_size = 0.2, random_state=2026)

best_lbol_model = Pipeline([
    ('impute', KNNImputer()),
    ('ratio', RatioGenerator(cols=flux_cols)),
    ('log', ColumnTransformer([('log_ratio', FunctionTransformer(func=np.log), ['F1100_over_F870', 'F1100_over_F500', 'F1100_over_F350', 'F1100_over_F250', 'F1100_over_F160', 'F1100_over_F70', 'F1100_over_F24', 'F1100_over_F12', 'F1100_over_F8', 'F870_over_F500', 'F870_over_F350', 'F870_over_F250', 'F870_over_F160', 'F870_over_F70', 'F870_over_F24', 'F870_over_F12', 'F870_over_F8', 'F500_over_F350', 'F500_over_F250', 'F500_over_F160', 'F500_over_F70', 'F500_over_F24', 'F500_over_F12', 'F500_over_F8', 'F350_over_F250', 'F350_over_F160', 'F350_over_F70', 'F350_over_F24', 'F350_over_F12', 'F350_over_F8', 'F250_over_F160', 'F250_over_F70', 'F250_over_F24', 'F250_over_F12', 'F250_over_F8', 'F160_over_F70', 'F160_over_F24', 'F160_over_F12', 'F160_over_F8', 'F70_over_F24', 'F70_over_F12', 'F70_over_F8', 'F24_over_F12', 'F24_over_F8', 'F12_over_F8'])], remainder='passthrough')),
    ('scale', RobustScaler()),
    ('model', RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1))
])

best_lbol_model.set_params(
model__bootstrap=True,
model__max_depth=30,
model__max_features=None,
model__min_samples_leaf=2,
model__min_samples_split=2,
model__n_estimators=1000
)

best_lbol_model_cv = cross_val_score(best_lbol_model, lbol_X_train, lbol_y_train, cv=cv, scoring='neg_root_mean_squared_error')
print(lbol_results['logCVscore'], -best_lbol_model_cv.mean())

0.03764870189432874 0.03764870189432869


In [20]:
best_lbol_model.fit(lbol_X_train, lbol_y_train)
lbol_y_preds = best_lbol_model.predict(lbol_X_test)
lbol_rmse = root_mean_squared_error(lbol_y_test, lbol_y_preds)
lbol_r2 = r2_score(lbol_y_test, lbol_y_preds)
print(lbol_rmse, lbol_r2)

0.014961946115345908 0.9999571929250277
